In [10]:
# 导入库，并用 d2l.synthetic_data 生成线性回归合成数据

import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

In [11]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [12]:
# 取出一个批次，查看数据格式

next(iter(data_iter))

[tensor([[ 0.4007, -0.6442],
         [-0.3357,  0.7173],
         [-0.6238, -0.4592],
         [ 0.2337,  0.5991],
         [ 1.3998, -1.1069],
         [-0.3160,  0.2983],
         [-2.0368,  0.3545],
         [ 0.3812, -0.7792],
         [-1.1557, -0.7459],
         [ 2.2800,  0.0082]]),
 tensor([[ 7.1927],
         [ 1.0839],
         [ 4.5086],
         [ 2.6312],
         [10.7787],
         [ 2.5460],
         [-1.0977],
         [ 7.5915],
         [ 4.4386],
         [ 8.7287]])]

In [13]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

In [14]:
# 手动初始化权重(正态分布)与偏置(置0)

net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [15]:
# 使用 PyTorch 内置均方误差损失

loss = nn.MSELoss()

In [16]:
# 使用内置 SGD 优化器

trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [17]:
# 训练循环：内置优化器自动完成 梯度清零 -> 反向传播 -> 参数更新

num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000376
epoch 2, loss 0.000101
epoch 3, loss 0.000101


In [18]:
# 输出估计参数与真实参数的误差

w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([ 0.0005, -0.0005])
b的估计误差： tensor([-0.0006])
